In [ ]:
from ASKpipeline import build_verification_graph
import pandas as pd
from collections import Counter
import json
import os

DOWNSAMPLE=True

In [ ]:
from typing import Dict, List
import pandas as pd

def run_verification_on_datasets(
    datasets: Dict[str, pd.DataFrame],
    question_col: str = "question",
    answer_col: str = "answer",
    build_graph_fn=None,
    show_progress: bool = True,
) -> Dict[str, List[str]]:
    """
    Runs the verification graph on each row of each dataframe.
    Returns: {dataset_name: [verdicts...]} in row order.
    """
    if build_graph_fn is None:
        from ASKpipeline import build_verification_graph
        build_graph_fn = build_verification_graph

    graph = build_graph_fn()
    results: Dict[str, List[str]] = {}

    for name, df in datasets.items():
        verdicts: List[str] = []
        iterator = df.itertuples(index=False)
        if show_progress:
            try:
                from tqdm import tqdm
                iterator = tqdm(list(iterator), desc=f"Verifying {name}")
            except Exception:
                pass

        for row in iterator:
            row_dict = row._asdict()
            state = {
                "question": row_dict.get(question_col, ""),
                "answer": row_dict.get(answer_col, ""),
            }
            try:
                out = graph.invoke(state)
                verdicts.append(out.get("verdict", "Not Found"))
            except Exception:
                verdicts.append("ERROR")

        results[name] = verdicts

    return results


In [ ]:
names = ['mintaka', 'qald', 'hotpot']

variants = ['small','base','large']
norm_files = [f'flan-t5-{variant}' for variant in variants]
vanilla_files = [f'vanilla-flan-t5-{variant}' for variant in variants]

files = norm_files + vanilla_files

file_name = files[0]


dataframes = {name: pd.read_csv(f'./LLM_answers/{file_name}/LLM_Answers_{name}.csv') for name in names}

if file_name.startswith('vanilla'):
    dataframes = {name: pd.read_csv(f'./LLM_answers/{file_name}/Vanilla_LLM_Answers_{name}.csv') for name in names}




In [ ]:
from ASKpipeline import build_verification_graph

state = {"queries": 'ASK WHERE {{ wd:Q5351150 wdt:P17 wd:Q49 . }UNION{ wd:Q49 wdt:P17 wd:Q5351150 . }}'}
graph = build_verification_graph(state)

row = dataframes["mintaka"].iloc[0]
state = {"question": row["SAE Question"], "answer": row["Answer"]}
out = graph.invoke(state)



In [ ]:
type(out['results'][0])

In [ ]:


all_results = {}

for file_name in files:
    if file_name.startswith('vanilla'):
        dataframes = {name: pd.read_csv(f'./LLM_answers/{file_name}/Vanilla_LLM_Answers_{name}.csv') for name in names}
    else: 
        dataframes = {name: pd.read_csv(f'./LLM_answers/{file_name}/LLM_Answers_{name}.csv') for name in names}
    

    file_results = {}
    for name, data in dataframes.items():
        print(f'Starting verification for {name}')
        if DOWNSAMPLE:
            print(f'Truncating {name} to 5 items')
            data = data[:5]
        row_results = []
        all_votes = []


        for _, row in data.iterrows():
            state = {'question': row["SAE Question"], "answer": row["Answer"]}
            print(f'state: \t{state}')
            out = graph.invoke(state)
            results = out.get("results") or []
            row_results.append([str(r) for r in results])

            all_votes.extend([str(r) for r in results])

        majority = Counter(all_votes).most_common(1)[0][0] if all_votes else "None"
        print(majority)

        fin_res = pd.DataFrame({
            "Results": row_results,
            "Majority": [majority] * len(row_results),
        })

        file_results[name] = fin_res
    out_dir = f'./final_results/{file_name}'
    os.makedirs(out_dir, exist_ok=True)

    # Save all results for this file_name to a single JSON file
    json_path = f'{out_dir}/{file_name}_results.json'
    file_results_json = {name: df.to_dict(orient='records') for name, df in file_results.items()}
    with open(json_path, 'w') as f:
        json.dump(file_results_json, f, indent=2)

    all_results[file_name] = file_results_json
    all_results_path = f'{out_dir}/all_results.json'
    all_results.to_json(all_results_path, orient="records", indent=2)

